# Day 4 Laboratory Exercise: Reading A Discharge Summary

This morning's walkthrough took these 200 discharge summaries apart. It showed
that the median one needs 968 tokens and that the model accepts 510, that a
pre-trained model's guesses tell you what it was trained on, that a training
loop is a loss and a gradient and a step, and that a falling loss says nothing
about a document the model has not seen.

It also asked a question the corpus could not answer, and the coded diagnoses
turned out not to be recoverable from the prose at all.

This afternoon you ask a question the corpus can answer. Every one of these
admissions ended somehow, and the doctor who wrote the summary wrote down how.
So the question is whether a model can read a discharge summary and tell you
whether the patient died.

That is extraction rather than prediction, and the distinction matters. You are
not asking a model to foresee a death. You are asking whether it can find a fact
that is already written down, which is exactly what the day's project means by
extracting statistics from text.

The obvious way to do it fails, and it fails in a way that looks like success.

## How To Work Through This

There are eight tasks. Each states a goal, gives you a cell marked
`# YOUR CODE HERE` naming the variables the check expects, and follows it with a
check cell that either confirms your answer or tells you what it wanted.

Task 1 asks you to write down two numbers and then set them aside. Do set them
aside. You will need them again at Task 7, and the point of the afternoon
depends on your having stopped thinking about them in between.

Tasks 5 and 8 each train a transformer three times. On a graphics processor that
is under a minute each and on a processor about three, so start the cell and read
ahead while it runs.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
import re

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers.utils import logging as hf_logging

DATA_DIR = Path.cwd().parent / "data"

RANDOM_STATE = 42

MODEL = "distilbert/distilbert-base-uncased"
REVISION = "12040accade4e8a0f71eabdb258fecc2e7e948be"

hf_logging.disable_progress_bar()
hf_logging.set_verbosity_error()
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 110)


def check(condition, success, failure):
    """Report whether a task was completed correctly."""
    print(success if condition else failure)


def pick_device():
    """Use a GPU when the image provides one, and fall back sensibly."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()

summaries = pd.read_csv(DATA_DIR / "discharge_summaries.csv")

# Tasks 1 and 7 both count tokens, so the tokeniser is made once here.
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)

print("Documents:", len(summaries))
print("PyTorch:", torch.__version__, "| training on:", DEVICE)

## The Tools You Are Given

The next cell holds the fine-tuning machinery for Tasks 5 and 8. Read it, run
it, and then leave it alone. Nothing in this laboratory exercise asks you to
change it, and both tasks that use it differ from each other by one argument.

Two things in it are worth noticing now.

`tokenise` takes a `truncation_side`. The walkthrough showed what that argument
does to a sequence of eight words. Here it decides what happens to a document of
nine hundred tokens.

`fine_tune` trains the same model three times, once per fold, and returns the
mean F1 on deaths, the mean accuracy, and the predictions and labels pooled
across the three held out sets. The pooled predictions are there because Task 6
needs to count something in them.

In [ ]:
FOLDS = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
EPOCHS = 3
BATCH_SIZE = 8


def tokenise(texts, truncation_side="right", max_length=512):
    """Encode every document, keeping one end when it does not fit."""
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL, revision=REVISION, truncation_side=truncation_side)
    return tokenizer(list(texts), truncation=True, max_length=max_length,
                     padding="max_length", return_tensors="pt")


def fine_tune(encoded, target, verbose=True):
    """Fine-tune the model once per fold and score the held out documents.

    Returns the mean F1 on the positive class, the mean accuracy, and the
    pooled held out predictions and labels.
    """
    labels = torch.tensor(target)
    scores, accuracies, pooled_pred, pooled_true = [], [], [], []

    for number, (train, test) in enumerate(FOLDS.split(target, target), 1):
        torch.manual_seed(RANDOM_STATE)
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL, revision=REVISION, num_labels=2).to(DEVICE)
        optimiser = torch.optim.AdamW(model.parameters(), lr=2e-5)
        generator = torch.Generator().manual_seed(RANDOM_STATE)

        model.train()
        for epoch in range(EPOCHS):
            order = train[torch.randperm(len(train),
                                         generator=generator).numpy()]
            for start in range(0, len(order), BATCH_SIZE):
                batch = order[start:start + BATCH_SIZE]
                optimiser.zero_grad()
                model(input_ids=encoded["input_ids"][batch].to(DEVICE),
                      attention_mask=encoded["attention_mask"][batch].to(DEVICE),
                      labels=labels[batch].to(DEVICE)).loss.backward()
                optimiser.step()

        model.eval()
        chunks = []
        with torch.no_grad():
            for start in range(0, len(test), 16):
                batch = test[start:start + 16]
                logits = model(
                    input_ids=encoded["input_ids"][batch].to(DEVICE),
                    attention_mask=encoded["attention_mask"][batch].to(DEVICE),
                ).logits
                chunks.append(logits.float().argmax(1).cpu().numpy())
        predicted = np.concatenate(chunks)

        scores.append(f1_score(target[test], predicted, zero_division=0))
        accuracies.append(accuracy_score(target[test], predicted))
        pooled_pred.append(predicted)
        pooled_true.append(target[test])
        if verbose:
            print(f"  fold {number}: f1 {scores[-1]:.3f}  "
                  f"accuracy {accuracies[-1]:.3f}")
        del model

    return (float(np.mean(scores)), float(np.mean(accuracies)),
            np.concatenate(pooled_pred), np.concatenate(pooled_true))


def cross_validate_texts(model, texts, target):
    """Fit a scikit-learn model on each fold and return mean f1 and accuracy."""
    scores, accuracies = [], []
    for train, test in FOLDS.split(texts, target):
        fitted = model.fit(texts[train], target[train])
        predicted = fitted.predict(texts[test])
        scores.append(f1_score(target[test], predicted, zero_division=0))
        accuracies.append(accuracy_score(target[test], predicted))
    return float(np.mean(scores)), float(np.mean(accuracies))


print("Fine-tuning machinery ready.")

## Task 1: Measure The Documents Against The Model

Before anything else, find out how much of each document the model will accept.
The walkthrough did this for the English summaries and you are going to need the
numbers yourself.

Build a tokeniser, count the tokens in every English summary, and store three
things:

- `median_tokens`, the median number of tokens per document,
- `n_too_long`, how many of the 200 documents need more than 510 tokens, and
- `median_kept`, the median fraction of a document that survives a 512 token
  limit, which is `min(tokens, 510) / tokens` for each document.

A `tokenizer` is already built for you in the first cell, and
`tokenizer.tokenize(text)` gives you the pieces for one document.

In [ ]:
# YOUR CODE HERE
# Count the tokens in every English summary, then store `median_tokens`,
# `n_too_long`, and `median_kept`.

In [ ]:
check(
    "median_tokens" in dir() and "n_too_long" in dir()
    and "median_kept" in dir()
    and abs(median_tokens - 968) < 5
    and n_too_long == 177
    and abs(median_kept - 0.53) < 0.02,
    "Task 1 complete. The median document needs 968 tokens, 177 of the 200 do "
    "not fit, and a 512 token limit keeps about 53 percent of a typical one. "
    "Write those down and then set them aside.",
    "Not right yet. Expected median_tokens near 968, n_too_long to be 177, and "
    "median_kept near 0.53.",
)

### Set Those Numbers Aside

Roughly half of a typical document cannot reach the model. Nothing will warn
you about it and nothing you do below will fail because of it.

Write the three numbers down, and then stop thinking about them. The next five
tasks are about a different question, and Task 7 will tell you when to come
back.

## Task 2: Work Out What The Outcome Column Says

The question is whether a patient died, so you need a column saying whether a
patient died. Look at what you have.

`outcome_en` holds seven distinct strings for a two-valued fact, and some of
them contain braces. `length_of_stay` is mostly numbers with a few oddities in
it. Neither the braces nor the oddities are a loading error.

Work out what the notation means before you rely on it. `data/SOURCES.md` names
the paper that published this dataset, and the answer is in there. You can also
work it out from the data, which is the more interesting route: compare a braced
value against what the summary text says about the same admission.

Then build the target:

- `died`, a numpy array of 200 zeros and ones, where one means the patient died,
- `n_deaths`, how many died, and
- `disagreement`, the `record_id` of the single admission whose outcome column
  contains both possibilities.

Treat a braced value as the one to believe.

In [ ]:
# YOUR CODE HERE
# Build `died`, `n_deaths`, and `disagreement` from the outcome column.

In [ ]:
check(
    "died" in dir() and "n_deaths" in dir() and "disagreement" in dir()
    and len(died) == 200
    and n_deaths == 62
    and disagreement == 53,
    "Task 2 complete. 62 of the 200 patients died, and record 53 is the one "
    "whose outcome column says both. Read the end of its summary before you "
    "move on.",
    "Not right yet. Expected `died` to have 200 entries, n_deaths to be 62, and "
    "disagreement to be 53.",
)

### Record 53

Its outcome column reads `alive {death}`, and its summary ends with the
sentence "death. evolves with refractory septic shock."

The hospital's database says the patient survived. The document says the patient
did not, and a physician reading the document corrected the database. That is
what a brace marks throughout this file.

Keep that in view for the rest of the afternoon. Extracting facts from clinical
text is not a workaround for when the database is unavailable. Here it is the
only place the correct value was written down.

## Task 3: Take A Number Out Of The Text

The other thing the agenda asks for is a statistic pulled out of the prose, so
pull one out. Every summary opens with an administrative preamble, and many of
them state the length of the admission in it, in a form like `stay: 06 d`.

Extract it and compare it against the structured column.

- `stated`, the number of summaries whose English text states a stay in days or
  hours,
- `comparable`, the number where the text states a stay in **days** and
  `length_of_stay` holds a plain number, so the handful of summaries measured
  in hours and the rows whose column is annotated both drop out, and
- `off_by_one`, how many of those `comparable` rows have a column value exactly
  one greater than the text value.

The pattern `r"stay:\s*(\d+)\s*(h|d)\b"` will find the field. Use
`re.search` on each summary.

In [ ]:
# YOUR CODE HERE
# Extract the stay from the text and compare it against `length_of_stay`,
# storing `stated`, `comparable`, and `off_by_one`.

In [ ]:
check(
    "stated" in dir() and "comparable" in dir() and "off_by_one" in dir()
    and stated == 109
    and comparable == 96
    and off_by_one == 47,
    "Task 3 complete. 109 summaries state a stay, 96 of those state it in days "
    "against a plain number in the column, and the two agree exactly 49 times "
    "and differ by one day the other 47. Nothing differs by more than a day.",
    "Not right yet. Expected stated to be 109, comparable to be 96, and "
    "off_by_one to be 47. Count only the summaries that state the stay in days "
    "and whose length_of_stay is a plain number.",
)

### Two Sources, One Day Apart

Of the 96 admissions where both sources give a figure in days, 49 agree exactly
and 47 have a column value one larger. Nothing differs by more than a day. That
is a counting convention rather than an error: one source counts the nights and
the other counts the days, including both the day of admission and the day of
discharge.

This is the ordinary situation with clinical text. The document and the database
mostly agree, they disagree in a way that follows a rule, and finding the rule is
the work. Record 53 was the case where the rule did not apply and the document
was simply right.

## Task 4: Build The Baseline First

You now have a target. Before reaching for a transformer, get a number from the
simplest thing that could possibly work, so that there is something to compare
against later. The walkthrough spent its last ten minutes on why this matters.

Build a scikit-learn pipeline of `TfidfVectorizer` and `LogisticRegression`,
score it with `cross_validate_texts`, and store the mean F1 on deaths in
`tfidf_f1` and the mean accuracy in `tfidf_accuracy`.

Use `make_pipeline`, pass `class_weight="balanced"` to the logistic regression
because the classes are uneven, and pass
`summaries["summary_en"].to_numpy()` as the texts.

Note that this method reads the whole document. It has no length limit at all.

In [ ]:
# YOUR CODE HERE
# Score a TF-IDF and logistic regression pipeline with cross_validate_texts,
# storing `tfidf_f1` and `tfidf_accuracy`.

In [ ]:
check(
    "tfidf_f1" in dir() and "tfidf_accuracy" in dir()
    and abs(tfidf_f1 - 0.816) < 0.05
    and abs(tfidf_accuracy - 0.905) < 0.04,
    "Task 4 complete. Counting words scores an F1 of about 0.82 and an accuracy "
    "of about 0.90, reading the whole document. That is the number to beat.",
    "Not right yet. Expected tfidf_f1 near 0.816 and tfidf_accuracy near 0.905 "
    "from a TF-IDF and logistic regression pipeline scored with "
    "cross_validate_texts.",
)

A bag of words, with no idea what a word means and no idea what order the words
came in, gets nine documents in ten right.

That is your baseline. Now do it properly.

## Task 5: Fine-tune A Transformer On It

Now the method the day is named after. DistilBERT has sixty-six million
pre-trained weights and knows something about English, which a word count does
not.

Encode the English summaries with `tokenise`, fine-tune with `fine_tune`, and
store the mean F1 in `bert_f1` and the mean accuracy in `bert_accuracy`. Keep
the predictions and labels it returns, because the next task needs them.

Use `tokenise` exactly as it comes, with no arguments beyond the texts. That is
what every tutorial does, and it is what the defaults give you.

This trains three times. Expect under a minute on a graphics processor and about
three on a processor.

In [ ]:
# YOUR CODE HERE
# Encode the summaries with `tokenise`, fine-tune with `fine_tune`, and store
# `bert_f1`, `bert_accuracy`, `bert_predicted`, and `bert_true`.

In [ ]:
check(
    "bert_f1" in dir() and "bert_accuracy" in dir()
    and "bert_predicted" in dir() and "bert_true" in dir()
    and bert_f1 < 0.3
    and bert_f1 < tfidf_f1 - 0.4,
    "Task 5 complete, and the result is worse than the bag of words by a wide "
    "margin. Do not fix it yet. Task 6 is about reading it correctly first.",
    "Not right yet. Expected `bert_f1` below 0.3 and at least 0.4 below "
    "tfidf_f1, from `tokenise` called with no arguments beyond the texts.",
)

### That Is Not What Was Supposed To Happen

Sixty-six million pre-trained weights, fine-tuned on this exact corpus for this
exact question, lost to counting words.

Before changing anything, read the two numbers you got rather than the one you
were hoping for.

## Task 6: Read The Accuracy Against A Baseline

The accuracy from Task 5 is around 0.69. On its own that sounds like a model
that has learned something imperfect.

Find out what it actually is.

- `base_rate`, the accuracy of always answering the commoner class, which is
  the larger of the two class proportions in `died`,
- `n_predicted_deaths`, how many of the pooled held out documents the model
  called a death, out of the 200 in `bert_predicted`, and
- `n_actual_deaths`, how many of them actually were, from `bert_true`.

In [ ]:
# YOUR CODE HERE
# Store `base_rate`, `n_predicted_deaths`, and `n_actual_deaths`.

In [ ]:
check(
    "base_rate" in dir() and "n_predicted_deaths" in dir()
    and "n_actual_deaths" in dir()
    and abs(base_rate - 0.69) < 0.005
    and n_actual_deaths == 62
    and n_predicted_deaths < 10
    and abs(bert_accuracy - base_rate) < 0.03,
    "Task 6 complete. The base rate is 0.69 and the model scored the base rate, "
    "because it found almost none of the 62 deaths. It did not learn something "
    "imperfect. It learned to say 'alive'.",
    "Not right yet. Expected base_rate near 0.69, n_actual_deaths to be 62, and "
    "n_predicted_deaths below 10.",
)

### The Accuracy Was The Base Rate

138 of the 200 patients survived, so a model that answers "alive" every time
scores 0.69. That is what this model scored, and it found almost none of the 62
deaths, which is why its F1 is near zero while its accuracy looks respectable.

So there was never a partly working model here. There was a model that learned
to answer with the commoner class, and an accuracy that concealed it.

That is the second time today a number has been read wrongly, and both times the
fix was the same: put a baseline next to it. In the walkthrough the baseline was
a bag of words that matched a fine-tuned transformer. Here it is a constant
answer that matches it.

Now, why? The transformer has strictly more information available than the bag
of words did. It knows what words mean and it knows what order they came in.
Something must be taking information away, and Task 7 is where you look for it.

## Task 7: Go Back To Task 1

Read your three numbers from Task 1 again, and then ask a question the
walkthrough deliberately did not answer: where in one of these documents does
the evidence of a death actually sit?

A discharge summary is written at the end of an admission and describes it in
the order it happened. So find out.

Among the documents where the patient died and the English text names it, using
the words `death`, `died`, or `deceased`, count:

- `visible_head`, how many have at least one of those words somewhere in
  their first 510 tokens, and
- `visible_tail`, how many have at least one of them somewhere in their last
  510 tokens.

A document shorter than 510 tokens counts in both, because all of it fits
either way.

Work in tokens rather than characters, because tokens are what the limit is
measured in. `tokenizer.tokenize(text)` gives you the list, and the words you
are looking for are single tokens in this vocabulary.

Then plot where the last mention falls, as a fraction of the way through each
document.

In [ ]:
# YOUR CODE HERE
# Count `visible_head` and `visible_tail` among the documents where the patient
# died and the text names it. Then plot where the last mention falls.

In [ ]:
check(
    "visible_head" in dir() and "visible_tail" in dir()
    and visible_head == 19
    and visible_tail == 61
    and visible_tail > visible_head * 3,
    "Task 7 complete. Reading from the front the model can see the answer in 19 "
    "of the 61 documents. Reading from the back it can see it in all 61. Now "
    "look again at what `tokenise` did for you in Task 5.",
    "Not right yet. Expected visible_head to be 19 and visible_tail to be 61, "
    "counted over the documents where the patient died and the text names it.",
)

### The Default Threw The Answer Away

61 documents out of 62 say in words that the patient died. 13 of those are
short enough to fit whole, so truncation costs them nothing. The other 48 have
to be cut, and only 6 of the 48 mention the death early enough to survive a cut
from the front. That is why the count from the front is 19 and the count from
the back is 61.

`truncation=True` keeps the beginning of a sequence, because `truncation_side`
defaults to `"right"`. The walkthrough showed that on eight words. On these
documents it means the model was handed the admission and the investigations,
and never reached the sentence saying how it ended.

So the model in Task 5 was not failing to learn. It was learning correctly from
evidence that had been removed before it ever saw it. Nothing raised an error,
nothing printed a warning, and the accuracy of 0.69 looked like a mediocre
result rather than a missing input.

## Task 8: Read The Other End

Fix it, and change nothing else.

Encode the same summaries keeping the end of each document instead of the
beginning, fine-tune with the same function on the same folds with the same
seed, and store the mean F1 in `tail_f1` and the mean accuracy in
`tail_accuracy`.

The only difference from Task 5 is one argument.

In [ ]:
# YOUR CODE HERE
# Encode keeping the end of each document, fine-tune, and store `tail_f1` and
# `tail_accuracy`.

In [ ]:
check(
    "tail_f1" in dir() and "tail_accuracy" in dir()
    and tail_f1 > 0.85
    and tail_f1 > bert_f1 + 0.5
    and tail_accuracy > tfidf_accuracy,
    "Task 8 complete. One argument took the F1 from near zero to above 0.85, "
    "past the bag of words, without touching the model, the seed, the folds, "
    "the learning rate, or the number of epochs.",
    "Not right yet. Expected tail_f1 above 0.85 and at least 0.5 above bert_f1, "
    "from `tokenise(..., truncation_side='left')` and the same `fine_tune` "
    "call as Task 5.",
)

### What Actually Happened

Nothing about the model changed. Same architecture, same sixty-six million
pre-trained weights, same seed, same three folds, same learning rate, same three
epochs, same 200 documents. One keyword argument moved, and the F1 went from
near zero to above 0.85.

Line the four numbers up and read them as a sequence.

Answering "alive" every time scores an accuracy of 0.69 and an F1 of zero. A bag
of words that reads the whole document scores about 0.90 and 0.82. A fine-tuned
transformer reading the first half of each document scores 0.69 and almost
nothing, which is to say it matched the constant answer. The same transformer
reading the last half of each document beats everything.

The ordering of those four is the whole afternoon. The most sophisticated method
came third, behind counting words and level with a constant, and it was not
because transformers are overrated. It was because it was reading the wrong half.

Three things are worth carrying out of this.

The first is that preprocessing decides what a model can possibly learn.
`truncation=True` is not a formatting detail, it is a choice about which half of
the evidence to discard, and the library makes that choice silently and
sensibly. Sensible defaults are chosen for short texts, and clinical documents
are not short.

The second is that a document has a shape. A discharge summary puts the outcome
at the end, a news article puts the conclusion at the top, a scientific paper
puts it in the abstract and again in the discussion, and a legal contract puts
the important part wherever it likes. Before truncating anything, ask where in
this kind of document the answer lives. There is no default that is right for
all of them.

The third is the habit, and it is the one that generalises furthest. Every
diagnostic step this afternoon came from putting a number next to another
number. Task 4's bag of words gave Task 5 something to be worse than. Task 6's
base rate turned an accuracy of 0.69 from a mediocre result into a model that
had learned nothing. Task 7's token positions turned that into a specific,
fixable cause. None of those came from looking harder at the transformer, and
none of them could have.

## Extensions

Work on these in any order if you have time left.

1. Task 8 keeps the last 510 tokens and throws away the beginning, which cannot
   be right either, since the diagnosis and the investigations are at the front.
   Split each document into two halves of 510 tokens, score each half on its
   own, and say what that tells you about where the information is.
2. There is a third option nobody used. Cut each document to its last 510
   tokens and cut it again to its first 510, then average the two models'
   predicted probabilities. Score it, and decide whether the extra complexity
   earned anything over Task 8.
3. A model with a longer limit would avoid the problem rather than working
   around it. Look up what `model_max_length` is for a model such as
   `allenai/longformer-base-4096` and work out from Task 1's token counts how
   many of these 200 documents it would take whole. Do not download it, because
   it is large and the session is short.
4. Search the text for the words `death`, `died`, and `deceased`, predict a
   death when any of them appears, and score that rule against the target with
   no model at all. Compare it against all four numbers in Task 8, then find the
   four documents it gets wrong and read them. Decide what the transformer is
   contributing over the search.
5. Repeat Task 8 on the Portuguese originals in `summary_pt`, keeping everything
   else the same. The tokeniser's vocabulary is English, so the same documents
   cost 2.56 tokens per word instead of 1.81 and fewer of them fit. Predict
   which way the score will move before you run it.
6. Every score in this notebook comes from three folds of 200 documents. Run
   Task 8 again with a different `random_state` in `FOLDS`, three times, and
   report the spread. Then decide which of this afternoon's conclusions survive
   that spread and which were never supported by it.

## What To Take Away

- A document is not a row, and the first thing to measure about one is whether
  the model can accept it. The median summary here needs 968 tokens and the
  model takes 510.
- `truncation=True` keeps the beginning. That is a decision about which evidence
  to discard, it is made silently, and no error or warning marks it.
- Ask where in a document the answer lives before cutting the document.
  Discharge summaries end with the outcome, and that is a property of the genre
  rather than of this dataset.
- An accuracy means nothing without the base rate beside it. 0.69 here was not a
  mediocre model, it was a model that answered "alive" every time.
- An F1 near zero beside a respectable accuracy is the signature of a model that
  has learned the commoner class. Print both.
- Fit the simplest thing first and keep its score. A bag of words beat a
  fine-tuned transformer, which was the evidence that something was wrong with
  the transformer's input rather than with the transformer.
- Braces in this dataset mark a physician overruling the database from the text.
  Record 53 is a patient the database recorded as discharged and the document
  recorded as dead.
- The most useful thing you can do to a number is put another number next to it.